In [1]:
import pandas as pd
import numpy as np
from snowflake.snowpark import Session


In [2]:
connection_parameters = {
    "account": "IJYGGJU-WI05238",
    "user": "Shank227",
    "password": "ShashankB2207$",
    "warehouse": "COMPUTE_WH",
    "database": "BANK_SEGMENTATION_DB",
    "schema": "PUBLIC",
    "role": "ACCOUNTADMIN"
}

In [3]:
session = Session.builder.configs(connection_parameters).create()

In [4]:
cluster_df = session.table("CUSTOMER_CLUSTERS").to_pandas()

In [5]:
df_encoded = pd.read_csv("encoded_dataset.csv")
df_encoded.head()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,False,True,False


In [6]:
df_encoded["Cluster"] = cluster_df["Cluster"]

df_encoded["Cluster"].value_counts()

Cluster
0    24071
2    11492
1     5625
Name: count, dtype: int64

In [7]:
numerical_profile = df_encoded.groupby("Cluster")[
    [
        "age",
        "duration",
        "campaign",
        "pdays",
        "previous"
    ]
].mean().round(2)

numerical_profile

,age,duration,campaign,pdays,previous
Cluster,,,,,
0,40.09,252.39,2.78,992.89,0.08
1,40.44,269.88,2.32,921.51,0.28
2,39.68,264.97,2.24,918.83,0.32


In [8]:
#job clusters
job_columns = [col for col in df_encoded.columns if col.startswith("job_")]

job_profile = df_encoded.groupby("Cluster")[job_columns].mean().T

job_profile

Cluster,0,1,2
job_blue-collar,0.217149,0.234311,0.235729
job_entrepreneur,0.039217,0.029156,0.030282
job_housemaid,0.027211,0.026844,0.022102
job_management,0.073491,0.069156,0.066655
job_retired,0.031947,0.058311,0.054212
job_self-employed,0.036849,0.028800,0.032370
job_services,0.092560,0.100089,0.102506
job_student,0.010801,0.031467,0.038113
job_technician,0.182336,0.141333,0.135660
job_unemployed,0.023638,0.023467,0.027236


In [9]:
#education clusters
education_columns = [col for col in df_encoded.columns if col.startswith("education_")]

education_profile = df_encoded.groupby("Cluster")[education_columns].mean().T

education_profile

Cluster,0,1,2
education_basic.6y,0.052885,0.058311,0.060129
education_basic.9y,0.141415,0.151644,0.155586
education_high.school,0.221927,0.238933,0.246171
education_illiterate,0.000499,0.000533,0.000261
education_professional.course,0.135516,0.113778,0.116690
education_university.degree,0.311703,0.272711,0.272450
education_unknown,0.039010,0.045689,0.046554


In [10]:
#martial clusters
marital_columns = [col for col in df_encoded.columns if col.startswith("marital_")]

marital_profile = df_encoded.groupby("Cluster")[marital_columns].mean().T

marital_profile

Cluster,0,1,2
marital_married,0.617590,0.593600,0.585016
marital_single,0.264592,0.292800,0.309085
marital_unknown,0.001828,0.002489,0.001914


In [11]:
housing_columns = [col for col in df_encoded.columns if col.startswith("housing_")]

housing_profile = df_encoded.groupby("Cluster")[housing_columns].mean().T

housing_profile

Cluster,0,1,2
housing_unknown,0.022309,0.026311,0.026540
housing_yes,0.529475,0.505067,0.521232


In [12]:
loan_columns = [col for col in df_encoded.columns if col.startswith("loan_")]

loan_profile = df_encoded.groupby("Cluster")[loan_columns].mean().T

loan_profile

Cluster,0,1,2
loan_unknown,0.022309,0.026311,0.026540
loan_yes,0.153587,0.144000,0.151497


In [13]:
contact_columns = [col for col in df_encoded.columns if col.startswith("contact_")]

contact_profile = df_encoded.groupby("Cluster")[contact_columns].mean().T

contact_profile

Cluster,0,1,2
contact_telephone,0.269453,0.6032,0.449443


In [14]:
cluster_names = {
    0: "Low Engagement Professionals",
    1: "Previously Engaged Customers",
    2: "Developing Customer Segment"
}

In [15]:
df_encoded["Segment"] = df_encoded["Cluster"].map(cluster_names)

In [16]:
df_encoded.head()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,Cluster,Segment
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,True,False,False,False,True,False,1,Previously Engaged Customers
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,True,False,False,False,True,False,2,Developing Customer Segment
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,True,False,False,False,True,False,2,Developing Customer Segment
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,True,False,False,False,True,False,1,Previously Engaged Customers
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,False,False,True,False,False,False,True,False,2,Developing Customer Segment


In [17]:
df_encoded["Cluster"].value_counts()

Cluster
0    24071
2    11492
1     5625
Name: count, dtype: int64